# **QR codes**

<div style="color:#777777;margin-top: -15px;">
<b>Author</b>: Norman Juchler |
<b>Course</b>: ADLS ISP |
<b>Version</b>: v1.0 <br><br>
<!-- 11.05.2026, v1.0: First version -->
</div>

QR codes are two-dimensional barcodes designed for fast and reliable encoding of information such as text, URLs, or binary data. They consist of a structured pattern of black and white modules, along with dedicated regions that enable robust detection, orientation, and error correction. Thanks to their resilience and ease of use, QR codes are widely used in everyday applications and can be scanned by most smartphone cameras.

In this notebook, you will explore both the **generation** and **decoding** of QR codes using Python. Specifically, you will learn how to create QR codes from input data and how to detect and decode them from images using computer vision techniques. The goal is to provide a hands-on understanding of how QR codes work in practice.

<br>

<figure style="text-align: center;">
  <img src="../data/doc/qr-code-sample.svg" alt="Sample QR code." width="30%" style="outline: 1px solid black; padding: 10px 10px 10px 10px;"/>
  <figcaption style="text-align: left; width: 30%; margin: 5px auto;">
    <b>Figure</b>: Example QR code — let's explore what information it encodes.
  </figcaption>
</figure>

---

## **Preparations**

The usual preparations... The package `isp` provides some helper functions to easily render images in this Jupyter notebook.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv2
import PIL

# Jupyter / IPython configuration:
# Automatically reload modules when modified, if the extension is available
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

# Enable vectorized output (for nicer plots)
%config InlineBackend.figure_formats = ["svg"]

# Inline backend configuration
%matplotlib inline

# Enable this line if you want to use the interactive widgets
# It requires the ipympl package to be installed.
#%matplotlib widget

import sys
sys.path.insert(0, "../")
import isp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## **Decode**

In this section, we demonstrate how QR codes can be automatically detected and decoded from images using computer vision techniques. OpenCV provides a built-in [QRCodeDetector](https://docs.opencv.org/4.x/de/dc3/classcv_1_1QRCodeDetector.html) that locates the QR code within an image and extracts its encoded content. The following example applies this process to sample images and visualizes the detected region. 


In [ ]:
# Load and process multiple sample images
filenames = [
    "../data/images/qr-code-sample-1.png",
    "../data/images/qr-code-sample-2.png",
    "../data/images/qr-code-sample-3.jpeg",
]

# Initialize QR code detector
detector = cv2.QRCodeDetector()

for fname in filenames:
    image = cv2.imread(fname, cv2.IMREAD_COLOR)
    if image is None:
        print(f"Failed to load {fname}")
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Detect and decode
    data, bbox, _ = detector.detectAndDecode(image_rgb)

    # Output result
    if bbox is not None and data:
        print(f"Decoded content from {fname}:", data)

        # Optional: draw bounding box
        pts = bbox.astype(int).reshape(-1, 2)
        for i in range(len(pts)):
            cv2.line(image_rgb, tuple(pts[i]), tuple(pts[(i + 1) % len(pts)]), (0, 255, 0), 2)

        isp.show_image(image_rgb, title=f"QR code detected in {fname}", 
                       suppress_info=True)
    else:
        print(f"No QR code detected in {fname}")


error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


----

## **Encode**

Now, let's see how to generate QR codes programmatically using OpenCV's [QRCodeEncoder](https://docs.opencv.org/4.x/d2/dbb/classcv_1_1QRCodeEncoder.html). Given an input string, the encoder converts the data into a structured QR code image. Since OpenCV produces a low-resolution output, we additionally resize the image for better visualization while preserving its sharp structure.

In [ ]:
encoder = cv2.QRCodeEncoder_create()

data = "I hope you enjoy this!"
qr_image = encoder.encode(data)
qr_resized = cv2.resize(qr_image, (300, 300), 
                        interpolation=cv2.INTER_NEAREST)

isp.show_image(qr_resized, title="Generated QR code", suppress_info=True)